# RE_205 · Quantes bicis caldran a les 8?

**Bloc 2 · Supervisat · repte 5 de 6 · durada: 2 sessions de 2 h**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes per a un servei de bicicleta compartida, com el Bicing. Cada nit, l'equip de logística
redistribueix les bicis entre estacions, i per fer-ho bé necessita saber **quantes se'n llogaran cada
hora del dia següent**. Tens dos anys d'històric amb el temps que feia i el tipus de dia.

La pregunta: **quin model preveu millor els lloguers per hora, i com l'has d'avaluar perquè l'error
que prometis sigui el que hi haurà l'any que ve?**

L'enunciat complet, amb el diccionari de columnes, és a [RE_205](../RE_205_bicis.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

## Què necessites saber abans

- Un model lineal és un vector i una multiplicació: [MA_01 §7](../../03_matematiques/MA_01_algebra_lineal.ipynb)
- Minimitzar un error per trobar els pesos: [MA_02 §3-§4](../../03_matematiques/MA_02_descens_gradient.ipynb)
- Arbres i boscos: [ML_02](../../01_teoria/ML_02_arbres.ipynb) i [ML_03](../../01_teoria/ML_03_boscos.ipynb)
- `train_test_split` i el seu parany: [EX_00 §1](../../02_practica/EX_00_caixa_eines.ipynb)
- `OneHotEncoder` i `ColumnTransformer`: [EX_00 §4 i §6](../../02_practica/EX_00_caixa_eines.ipynb)
- MAE i $R^2$: si no has fet el [RE_204](../RE_204_formigo.ipynb), mira-hi la caixa d'eines.

## Les dades

**Bike Sharing** (Fanaee-T i Gama, 2013; UCI), a OpenML amb `data_id=42712`. Llicència CC BY 4.0. Són
dades del sistema Capital Bikeshare de **Washington DC**, 2011 i 2012. Cada fila és **una hora**:
estació de l'any, any (`year` = 0 per al 2011, 1 per al 2012), mes, hora, si és festiu o laborable,
el temps (`weather`), la temperatura real i la sensació tèrmica (en °C), la humitat (0-1) i el vent.
L'objectiu és `count`, el nombre de bicis llogades aquella hora.

Documentació: <https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=42712, as_frame=True).frame
print(df.shape)
df.head()

## Nivell 1 · Explorar

### 1.1 · Què hi ha a la taula

Forma, tipus, rangs i nuls. Quantes hores hi ha de cada any? Si cada any té 365 dies de 24 hores, en falten? Quines columnes són categòriques, i quines són numèriques però potser no s'haurien de tractar com a tals?

In [ ]:
print(df.dtypes)
print(df.describe().T[["min", "max", "mean"]].round(2))
per_any = df["year"].value_counts().sort_index()
assert per_any.sum() == 17379
print(per_any, "\nfalten:", 24 * 365 - per_any[0], "hores al 2011 i", 24 * 366 - per_any[1], "al 2012")

> **Resultat.** 8.645 hores el 2011 i 8.734 el 2012: falten 115 i 50 hores (hores sense cap lloguer o sense registre). `month`, `hour` i `weekday` són números però cíclics: l'hora 23 és al costat de la 0.

### 1.2 · La forma del dia

Fes el gràfic de la mitjana de lloguers per hora del dia, amb una línia per als dies laborables i una altra per als no laborables. Què et diu sobre qui fa servir les bicis? Podria una recta capturar aquesta relació amb l'hora?

In [ ]:
t = df.pivot_table(index="hour", columns="workingday", values="count", aggfunc="mean", observed=True)
assert t.shape == (24, 2)
t.plot(marker="o"); plt.ylabel("lloguers per hora (mitjana)"); plt.show()
t.round(0).T

> **Resultat.** Laborables: dos pics (8 h i 17-18 h), de gent que va a treballar. No laborables: una campana al migdia. Cap recta no captura cap de les dues formes; aquest gràfic és el que justifica tota la segona meitat del repte.

### 1.3 · El temps

Quantes hores hi ha de cada valor de `weather`? Hi ha alguna categoria tan rara que pugui donar problemes en partir les dades? Mira també si hi ha valors de `humidity` o `windspeed` que no tinguin sentit físic.

In [ ]:
vc = df["weather"].value_counts()
assert vc.sum() == len(df)
print(vc)
print(df.loc[df["weather"] == vc.index[-1], ["year", "month", "hour", "count"]])
print("humitat 0:", (df["humidity"] == 0).sum(), "| vent 0:", (df["windspeed"] == 0).sum())

> **Resultat.** `heavy_rain` només té 3 files (una el 2011 i dues el 2012): en una partició pot no aparèixer a l'entrenament, i el one-hot ha de saber-ho gestionar. Hi ha 22 hores amb humitat 0 (impossible a Washington; és un error del sensor) i 2.180 amb vent 0 (probablement «no mesurat» o calma).

## Nivell 2 · Respondre

### 2.1 · Una partició honesta

El model s'ha de fer servir l'any que ve. Parteix les dades **pel temps**: entrenament = 2011, test = 2012. Prepara `X` sense `count` (i decideix què fas amb `year`) i `y = count`. Fes servir un `ColumnTransformer` que codifiqui les categòriques amb one-hot i deixi passar les numèriques.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

cat = ["season", "holiday", "workingday", "weather"]
num = ["month", "hour", "weekday", "temp", "feel_temp", "humidity", "windspeed"]
d = df.copy()
d[cat] = d[cat].astype(str)
tr, te = d[d["year"] == 0], d[d["year"] == 1]
X_train, y_train = tr[cat + num], tr["count"]
X_test, y_test = te[cat + num], te["count"]
assert (te["year"] == 1).all() and len(tr) + len(te) == 17379

pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat)], remainder="passthrough")
Xt = pre.fit_transform(X_train)
assert not np.isnan(np.asarray(Xt, dtype=float)).any()
print(X_train.shape, X_test.shape, "->", Xt.shape)

> **Resultat.** `year` es treu: a l'entrenament val sempre 0, i el model no en pot aprendre res. Queden 11 columnes que es converteixen en 19 (amb el one-hot).

### 2.2 · La teoria, al codi: la recta i el seu error

Entrena una `LinearRegression` amb aquest preprocés (l'hora com a número). Comprova que `predict` és exactament $X\,w + b$ calculant-ho tu amb la matriu transformada. Després implementa el MAE i l'RMSE i compara'ls amb scikit-learn. Compara'ls també amb un «model» que prediu sempre la mitjana de l'entrenament.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.base import clone   # cada pipeline amb el seu preprocés: make_pipeline no en fa còpia
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lin = make_pipeline(clone(pre), LinearRegression()).fit(X_train, y_train)
Xte = np.asarray(lin[0].transform(X_test), dtype=float)
p_ma = Xte @ lin[-1].coef_ + lin[-1].intercept_
assert np.allclose(p_ma, lin.predict(X_test))

def mae(yv, p): return np.mean(np.abs(np.asarray(yv) - p))
def rmse(yv, p): return np.sqrt(np.mean((np.asarray(yv) - p) ** 2))

p = lin.predict(X_test)
assert np.isclose(mae(y_test, p), mean_absolute_error(y_test, p))
assert np.isclose(rmse(y_test, p), np.sqrt(mean_squared_error(y_test, p)))
assert rmse(y_test, p) >= mae(y_test, p)
pm = np.full(len(y_test), y_train.mean())
print(f"recta:   MAE {mae(y_test, p):.1f}  RMSE {rmse(y_test, p):.1f}  R2 {r2_score(y_test, p):.3f}")
print(f"mitjana: MAE {mae(y_test, pm):.1f}  RMSE {rmse(y_test, pm):.1f}  R2 {r2_score(y_test, pm):.3f}")

> **Resultat.** La recta (MAE 129,9, RMSE 191,0, $R^2$ 0,164) millora la mitjana (MAE 168,3) però poc. El $R^2$ de la mitjana de 2011 és **negatiu** al 2012 (−0,189): la mitjana d'hora del 2011 és 144 i la del 2012, 235; el 2012 es van llogar moltes més bicis, i la mitjana de l'any anterior es queda curta. És el primer avís que el temps importa.

### 2.3 · Com fer servir l'hora

Prova tres maneres d'aprofitar l'hora i compara'n el MAE de test: (a) la recta amb l'hora com a número (l'2.2), (b) la recta amb l'hora **en one-hot** (24 columnes), i (c) un arbre o un bosc amb l'hora com a número. Quin model aprofita millor l'hora, i per què? Relaciona-ho amb el gràfic de l'1.2.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

cat_h = cat + ["hour"]
num_h = [c for c in num if c != "hour"]
Xtr_h, Xte_h = X_train.copy(), X_test.copy()
Xtr_h["hour"], Xte_h["hour"] = Xtr_h["hour"].astype(str), Xte_h["hour"].astype(str)
pre_h = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_h)], remainder="passthrough")
lin_h = make_pipeline(pre_h, LinearRegression()).fit(Xtr_h, y_train)
M = pd.DataFrame(np.asarray(lin_h[0].transform(Xtr_h).todense() if hasattr(lin_h[0].transform(Xtr_h), "todense")
                            else lin_h[0].transform(Xtr_h)), columns=lin_h[0].get_feature_names_out())
assert (M.filter(like="cat__hour_").sum(axis=1) == 1).all()

bosc = make_pipeline(clone(pre), RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)).fit(X_train, y_train)
res = {"(a) recta, hora número": lin.predict(X_test),
       "(b) recta, hora one-hot": lin_h.predict(Xte_h),
       "(c) bosc, hora número": bosc.predict(X_test)}
for k, v in res.items():
    print(f"{k:26s} MAE {mae(y_test, v):6.1f}  R2 {r2_score(y_test, v):.3f}")

> **Resultat.** MAE (a) 129,9, (b) 109,5 i (c) 88,8 ($R^2$ 0,16, 0,43 i 0,64). El one-hot de l'hora millora molt la recta (pot fer un pes per hora i dibuixar els pics), i el bosc, amb l'hora com a número, encara la supera, perquè talla l'hora en trossos i combina hora i tipus de dia.

### 2.4 · Aleatori contra temporal

Entrena el **mateix** bosc de l'2.3 amb una partició **aleatòria** (50 % / 50 %, `random_state=42`) de totes les files, amb les mateixes columnes. Compara el MAE amb el de la partició temporal. Quina xifra posaries a l'informe per a logística, i per què?

In [ ]:
from sklearn.model_selection import train_test_split

Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(d[cat + num], d["count"], test_size=0.5, random_state=42)
print(len(Xa_te), len(X_test))
bosc_a = make_pipeline(clone(pre), RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)).fit(Xa_tr, ya_tr)
print(f"aleatòria: MAE {mae(ya_te, bosc_a.predict(Xa_te)):.1f} | temporal: MAE {mae(y_test, bosc.predict(X_test)):.1f}")

> **Resultat.** La partició aleatòria dona la meitat d'error (MAE 45,0 contra 88,8). És enganyós: el model ha vist dies veïns dels del test, i no ha hagut de preveure el creixement de 2012. La xifra honesta per a logística és la temporal.

### 2.5 · On s'equivoca

Amb el millor model de la partició temporal, calcula l'error (real − predit) de cada hora de test i fes-ne la mitjana per mes. L'error té signe? Què et diu sobre el que el model no sap?

In [ ]:
err = y_test - bosc.predict(X_test)
per_mes = err.groupby(X_test["month"]).mean()
assert len(per_mes) == 12
assert np.isclose((per_mes * X_test["month"].value_counts().sort_index()).sum() / len(err), err.mean())
per_mes.plot.bar(); plt.axhline(0, color="k", lw=0.8); plt.ylabel("real − predit"); plt.show()
print("error mitjà global:", round(err.mean(), 1))

> **Resultat.** L'error mitjà és +83,6 i és positiu **tots** els mesos (de +51 al desembre a +106 al març): el model es queda curt sistemàticament, perquè el 2012 el servei va créixer i el model només ha vist el 2011. Cap columna no explica aquest creixement; una solució seria afegir la tendència, o reentrenar amb dades recents.

## Nivell 3 · Més enllà

### 3.1 · L'hora en un cercle

Escriu l'hora com a dues columnes, $\sin(2\pi h/24)$ i $\cos(2\pi h/24)$, i torna a entrenar la recta. Millora respecte de l'hora com a número? I respecte del one-hot? Per què creus que passa?

In [ ]:
def cicle(X):
    X = X.copy()
    X["hour_sin"] = np.sin(2 * np.pi * X["hour"] / 24)
    X["hour_cos"] = np.cos(2 * np.pi * X["hour"] / 24)
    return X.drop(columns="hour")
lin_c = make_pipeline(clone(pre), LinearRegression()).fit(cicle(X_train), y_train)
print(f"recta amb sin/cos: MAE {mae(y_test, lin_c.predict(cicle(X_test))):.1f}")

> **Resultat.** Sin/cos (MAE 116,9) millora l'hora com a número (129,9), però no arriba al one-hot (109,5): un sol cicle (una ona al dia) no pot fer els **dos** pics dels laborables.

### 3.2 · Laborable × hora

El gràfic de l'1.2 diu que l'hora no vol dir el mateix en un laborable que en un festiu. Crea una columna que combini les dues coses (per exemple, `"1_8"` per a laborable a les 8) i fes-la servir en one-hot a la recta. Quant s'acosta al bosc?

In [ ]:
def combina(X):
    X = X.copy()
    X["wd_hour"] = X["workingday"].astype(str) + "_" + X["hour"].astype(str)
    return X.drop(columns="hour")
pre_c = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat + ["wd_hour"])], remainder="passthrough")
lin_wd = make_pipeline(pre_c, LinearRegression()).fit(combina(X_train), y_train)
print(f"recta amb laborable×hora: MAE {mae(y_test, lin_wd.predict(combina(X_test))):.1f} | bosc: {mae(y_test, bosc.predict(X_test)):.1f}")

> **Resultat.** La recta amb la interacció baixa a MAE 100,3 i s'acosta al bosc (88,8): bona part del que el bosc «sabia» era aquesta combinació.

## Conclusió

El millor model és el bosc, amb un MAE d'unes 89 bicis per hora quan s'entrena amb el 2011 i es
prova amb el 2012. La recta amb l'hora com a número es queda a 130, i amb l'hora en one-hot a 110:
els laborables tenen dos pics (8 h i 17-18 h) que una recta amb un sol pes per a l'hora no pot
dibuixar. La partició aleatòria donaria la meitat d'error (45), però és enganyosa, perquè el model
veu dies veïns dels del test i no ha de preveure el futur. L'error té signe: el model es queda curt
tots els mesos del 2012 (+84 bicis per hora de mitjana), perquè el servei va créixer i només ha vist
el 2011. La xifra per a logística és la temporal, amb l'avís que cal reentrenar amb dades recents.